# PCA monitoring with T-squared and SPE

This notebook follows Days 18–20. It uses the chronological first 70% as a reference period and scores the remaining 30%. The reference split is an educational choice, not proof that this interval represents stable production. No outcome labels enter feature selection, preprocessing, PCA, or threshold calculation.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'data').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
from src.secom_data import load_secom
from src.multivariate import fit_pca_monitor, score_pca_monitor

## 1. Split by source order before fitting anything

The feature filter, median imputer, scaler, and PCA are all learned from the earlier reference rows only. Candidate features are chosen using only baseline missingness and variance, not pass/fail outcomes.

In [ ]:
data = load_secom(ROOT / 'data' / 'raw')
cut = int(len(data.features) * 0.70)
baseline = data.features.iloc[:cut]
later = data.features.iloc[cut:]
monitor = fit_pca_monitor(baseline, max_missing_fraction=0.20, variance_retained=0.90, threshold_quantile=0.99)
scored = score_pca_monitor(monitor, later)
pca = monitor.pipeline.named_steps['pca']
print(f'Baseline observations: {len(baseline)}')
print(f'Later observations scored: {len(later)}')
print(f'Features selected from baseline: {len(monitor.features)}')
print(f'PCA components retained: {pca.n_components_}')
print(f'Cumulative variance retained: {pca.explained_variance_ratio_.sum():.1%}')
print(f'Exploratory baseline T² threshold: {monitor.t2_threshold:.3f}')
print(f'Exploratory baseline SPE threshold: {monitor.spe_threshold:.3f}')
print(f'Later T² flags: {int(scored.t2_signal.sum())}; SPE flags: {int(scored.spe_signal.sum())}; either: {int(scored.investigate.sum())}')

## 2. Inspect scores and loadings

Scores locate observations in component space. Loadings show which anonymous measurements contribute to each axis. Names remain neutral because the source does not identify the physical sensors.

In [ ]:
baseline_scored = score_pca_monitor(monitor, baseline)
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(baseline_scored.pc1, baseline_scored.pc2, s=12, alpha=0.35, label='Reference period')
ax.scatter(scored.pc1, scored.pc2, c=np.where(scored.investigate, '#d1495b', '#00798c'), s=16, alpha=0.7, label='Later period (red = investigate)')
ax.set(xlabel='Principal component 1', ylabel='Principal component 2', title='Exploratory PCA score view')
ax.legend(); ax.grid(alpha=0.2); plt.show()

loadings = pd.DataFrame(pca.components_[:2].T, index=monitor.features, columns=['PC1_loading', 'PC2_loading'])
print('Largest absolute loadings for PC1:')
display(loadings.reindex(loadings.PC1_loading.abs().sort_values(ascending=False).head(10).index))
print('Largest absolute loadings for PC2:')
display(loadings.reindex(loadings.PC2_loading.abs().sort_values(ascending=False).head(10).index))

## 3. View multivariate monitoring statistics over later observations

T-squared measures distance within retained components; SPE measures the part not represented by those components. The empirical 99th percentile thresholds are exploratory and do not guarantee a chosen production false-alarm rate.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(scored.index, scored.t2, linewidth=0.8)
axes[0].axhline(monitor.t2_threshold, color='crimson', linestyle='--', label='Baseline empirical threshold')
axes[0].set_ylabel('T-squared'); axes[0].legend()
axes[1].plot(scored.index, scored.spe, linewidth=0.8)
axes[1].axhline(monitor.spe_threshold, color='crimson', linestyle='--', label='Baseline empirical threshold')
axes[1].set_ylabel('SPE / Q'); axes[1].set_xlabel('Original row index'); axes[1].legend()
for ax in axes: ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()
scored.loc[scored.investigate].head(10)

## Interpretation prompts

1. What distinguishes high T-squared from high SPE?
2. Why are these flags only investigation prompts?
3. What possible issue arises if the earlier period already contains shifts?
4. What additional data (tool, recipe, lot, measurement context) would help investigate an observation?
5. Why is PCA not a causal explanation?